In [ ]:
##########
# SETUP
##########

import sys, os
sys.path.append(os.path.abspath('..'))
sys.path.append('.')
from utils import loadFSL, mkdir_no_exist, FSLeyesServer

import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')
loadFSL()

from fsl.wrappers import bet, fast, flirt
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn import plotting
import os.path as op
import subprocess
import glob
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

In [ ]:
########
# PATHS
########

base = os.path.abspath("")
t1_path     = os.path.join(base, 'T1', 'T1w.nii.gz')
func_path   = os.path.join(base, 'fMRI', 'tfMRI_LANGUAGE_LR', 'tfMRI_LANGUAGE_LR.nii.gz')
events_path = os.path.join(base, 'fMRI', 'tfMRI_LANGUAGE_LR', 'tfMRI_LANGUAGE_LR_task_onsets.csv')

out_dir = os.path.join(base, 'derivatives')
mkdir_no_exist(out_dir)

In [ ]:
%gui wx

In [ ]:
fsleyesDisplay = FSLeyesServer()
fsleyesDisplay.show()

In [ ]:
# Looking at the T1 image

fsleyesDisplay.load(t1_path)

In [ ]:
#################
# Skull Stripping
#################

t1_path_strip = op.join(out_dir, 'T1w_brain')
subprocess.run(['bet', t1_path, t1_path_strip, '-m', '-R'])

In [ ]:
# Looking at stripped brain for QC

fsleyesDisplay.load(t1_path_strip)

In [ ]:
##############
# Segmentation
##############

t1_path_seg = op.join(out_dir, 'T1w_fast')
fast_target = t1_path_strip
fast(imgs=[fast_target], out=t1_path_seg, n_classes = 3)

In [ ]:
# Display Segmentation and do QC

t1_path_seg = op.join(out_dir, 'T1w_fast')

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(t1_path_strip)
fsleyesDisplay.load(glob.glob(t1_path_seg + '*pve_0*')[0])
fsleyesDisplay.load(glob.glob(t1_path_seg + '*pve_1*')[0])
fsleyesDisplay.load(glob.glob(t1_path_seg + '*pve_2*')[0])
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[1]).cmap = 'Red'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[2]).cmap = 'Green'
fsleyesDisplay.displayCtx.getOpts(fsleyesDisplay.overlayList[3]).cmap = 'Blue'

In [ ]:
###############
# Registration
###############

mni_template = op.expandvars(op.join('$FSLDIR', 'data', 'standard', 'MNI152_T1_1mm_brain'))

source = t1_path_strip 
reference = mni_template 
result = op.join(out_dir, 'T1w_brain_2_mni')
flirt(source, reference, out=result, omat=op.join(out_dir, 'T1w_brain_2_mni.mat')) 
# save the structural -> MNI matrix to apply it later to fMRI data

In [ ]:
# Dislplay registration results for QC

fsleyesDisplay.resetOverlays()
fsleyesDisplay.load(reference) 
fsleyesDisplay.load(result)

In [ ]:
mask = t1_path_strip + '_mask.nii.gz'
cuts = (0, -18, 20)     # same slices in every panel (x, y, z in mm)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), facecolor='black')

plotting.plot_anat(t1_path, cut_coords=cuts, axes=axes[0])

plotting.plot_anat(t1_path_strip + '.nii.gz', cut_coords=cuts, axes=axes[1])

d = plotting.plot_anat(t1_path, cut_coords=cuts, axes=axes[2])
d.add_contours(mask, colors='red', linewidths=1)

fig.savefig(op.join(fig_dir, 'skullstrip.png'), dpi=300,
            bbox_inches='tight', facecolor='black')

# Claude code

In [ ]:
seg = t1_path_seg + '_seg.nii.gz'
colors = ['green', 'blue', 'red']            # CSF, GM, WM
cmap = ListedColormap(colors)

fig = plt.figure(figsize=(13, 5), facecolor='black')
d = plotting.plot_roi(seg, bg_img=t1_path_strip + '.nii.gz', cmap=cmap,
                      vmin=1, vmax=3, alpha=0.8, figure=fig,)
fig.legend(handles=[Patch(color=c, label=l) for c, l in
                    zip(colors, ['CSF', 'Grey matter', 'White matter'])],
           loc='lower center', ncol=3, frameon=False, labelcolor='white')
fig.savefig(op.join(fig_dir, 'segmentation.png'), dpi=300, bbox_inches='tight', facecolor='black')

# Claude code

In [ ]:
fig_dir = op.join(base, 'figures')
mkdir_no_exist(fig_dir)

d = plotting.plot_anat(reference + '.nii.gz')
d.add_edges(t1_path_strip + '.nii.gz')
d.savefig(op.join(fig_dir, 'reg_before.png'), dpi=300)

d = plotting.plot_anat(reference + '.nii.gz')
d.add_edges(result + '.nii.gz')
d.savefig(op.join(fig_dir, 'reg_after.png'), dpi=300)

# Claude code